# LOCI Infineon TriCore (AURIX) execution-time model

Deploys the model package from AWS Marketplace to a SageMaker endpoint, sends assembly blocks from `../data/tricore_dataset.csv` and reads back the prediction for each block.

The input is a CSV with one column, `r.asm`; each row is one assembly basic block (one instruction per line inside the quoted cell).

The response is a NumPy array with one row per input block and four columns:

| column | meaning |
|---|---|
| 0 `std` | predicted standard deviation of the execution time (cycles) |
| 1 `mean` | predicted mean execution time (cycles) |
| 2 `fd_score` | out-of-distribution score: larger means the block is further from the training data and the prediction is less reliable |
| 3 `threshold` | calibrated cut-off for `fd_score` (constant per model). Rows with `fd_score` above it should be treated with caution |

A subset of columns can be requested through the `predicted_outputs` field of the custom-attributes header, e.g. `{"predicted_outputs": ["std", "mean"]}` for the two-column form.


In [ ]:
import io
import json

import boto3
import matplotlib.pyplot as plt
import numpy
import pandas
import sagemaker
from scipy.stats import norm


In [ ]:
# Running locally: set the IAM role SageMaker should assume
# role = "arn:aws:iam::<account>:role/<sagemaker-execution-role>"

# Running inside a SageMaker notebook: use the notebook's role
# role = sagemaker.get_execution_role()

sagemaker_session = sagemaker.Session()
runtime = boto3.client("runtime.sagemaker")

endpoint_name = "loci-at-1m-endpoint"
# Model package ARN from your AWS Marketplace subscription
modelpackage_arn = ""

model = sagemaker.ModelPackage(
    role=role,
    model_package_arn=modelpackage_arn,
    sagemaker_session=sagemaker_session,
)


In [ ]:
# Deploy the model (takes a few minutes)
predictor = model.deploy(1, "ml.g5.xlarge", endpoint_name=endpoint_name)


In [ ]:
data = pandas.read_csv('../data/tricore_dataset.csv')
print(f"{len(data)} blocks")
print(data['r.asm'].iloc[0])
data_bytes = data.to_csv(None, index=False)


In [ ]:
# Perform a prediction. Accept selects the response encoding; the array
# below is what ../example_outputs/ contains for this dataset.
response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="text/csv",
    Accept="application/x-npy",
    Body=data_bytes,
    # Optional: request a subset of columns
    # CustomAttributes=json.dumps({"predicted_outputs": ["std", "mean"]}),
)
predictions = numpy.load(io.BytesIO(response["Body"].read()))

cycle_std, cycle_mean, fd_score, threshold = predictions.T
result = pandas.DataFrame(
    {"mean_cycles": cycle_mean, "std_cycles": cycle_std,
     "fd_score": fd_score, "in_distribution": fd_score <= threshold}
)
print(f"fd_score threshold for this model: {threshold[0]:.3f}")
print(result.round(2).to_string())


In [ ]:
def plot_gaussian(mean, std_dev):
    x = numpy.linspace(mean - 4 * std_dev, mean + 4 * std_dev, 500)
    plt.plot(x, norm.pdf(x, loc=mean, scale=std_dev))
    plt.title('Predicted execution time, block 0')
    plt.xlabel('Cycles')
    plt.ylabel('Probability density')
    plt.grid(True)
    plt.show()


plot_gaussian(cycle_mean[0], cycle_std[0])


In [ ]:
# Delete the endpoint and the model
model.sagemaker_session.delete_endpoint(endpoint_name)
model.sagemaker_session.delete_endpoint_config(endpoint_name)
model.delete_model()
